# B6 · Explicabilité, équité et éthique — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b6_explicabilite/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b6_explicabilite/03_notebook.ipynb`, créée par `python tools/start_chapter.py B6`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B6.Q1 à B6.Q12, B6.R1 à B6.R3, B6.1, B6.3 à B6.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B6.12 à B6.17 | les modèles tabulaires : explications globales sur California, Shapley exact avec ta librairie, shap sur un gradient boosting, le rôle du fond, Penguins (*prochaine session de génération*) | | | |
| B | B6.18 à B6.22 | les réseaux : gradients intégrés et Grad-CAM dans ta librairie, attributions sur un CNN CIFAR-10, Grad-CAM par hooks sur le ResNet du B1, un Grad-CAM à réparer, le test de santé (*prochaine session de génération*) | | | |
| C | B6.23 à B6.29 | l'équité : tes indicateurs par groupe, l'audit d'Adult avec intervalles, l'effet du retrait du sexe, l'atténuation et son coût, la fiche modèle et le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B6")  # your own library

## Objectifs et rappel express

- Calculer à la main des valeurs de Shapley exactes, des gradients intégrés et une carte Grad-CAM, et les relier à ta librairie (`explain.py`) et à shap.
- Expliquer un modèle tabulaire globalement (permutation, dépendance partielle, ICE) et localement (Shapley, waterfall), en disant quel fond a servi.
- Produire des cartes d'attribution sur un CNN et les soumettre à un test de santé.
- Auditer un classifieur groupe par groupe, avec des intervalles, essayer des corrections, chiffrer leur coût et documenter le tout dans une fiche modèle.

**Rappel express.** Shapley : $v(S) = \frac{1}{B}\sum_k f(\mathbf{x}_S, \mathbf{b}^{(k)}_{\bar S})$, $\phi_i = \sum_{S \not\ni i} \frac{|S|!\,(M - |S| - 1)!}{M!}\,[v(S \cup \{i\}) - v(S)]$, $\sum_i \phi_i = f(\mathbf{x}) - v(\varnothing)$ ; modèle linéaire : $\phi_i = w_i(x_i - \bar b_i)$. Gradients intégrés : $\operatorname{IG}_i = (x_i - x'_i)\int_0^1 \partial_i f(\mathbf{x}' + \alpha(\mathbf{x} - \mathbf{x}'))\,d\alpha$, de somme $f(\mathbf{x}) - f(\mathbf{x}')$. Grad-CAM : $\alpha_c$ = moyenne spatiale de $\partial y^k / \partial A^c$, carte $\operatorname{ReLU}(\sum_c \alpha_c A^c)$. Équité : $\Delta_{\text{DP}}$ = écart des taux de sélection, $\Delta_{\text{EO}}$ = le plus grand des écarts de TPR et de FPR ; $\operatorname{FPR} = \frac{p}{1-p}\cdot\frac{1 - \operatorname{PPV}}{\operatorname{PPV}}\cdot(1 - \operatorname{FNR})$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B6.1, B6.3, B6.4 et B6.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un vecteur : une liste de nombres (`[0.5, -1.25]`). Un taux s'écrit comme une proportion (`0.25`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B6.2 et B6.6, l'estimation 🧮 B6.7, la réflexion (🗣️ B6.8, ⚖️ B6.9 et B6.10, 📄 B6.11) et l'entretien se corrigent avec `05_solutions.md`.

**B6.Q1 — Interprétable par construction ou expliqué après coup ?**

In [ ]:
# B6.Q1: the values you found on paper
answer_B6_Q1a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q1b = ...  # b) True or False
answer_B6_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_B6_Q1d = ...  # d) True or False
answer_B6_Q1e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B6_Q1a, answer_B6_Q1b, answer_B6_Q1c, answer_B6_Q1d, answer_B6_Q1e]):
    wb.check(f"B6.Q1{letter}", answer)

**B6.Q2 — Explication globale ou locale : quel outil répond à quelle question ?**

In [ ]:
# B6.Q2: the values you found on paper
answer_B6_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q2b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q2c = ...  # c) True or False
answer_B6_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_B6_Q2e = ...  # e) a number, 2 decimals

for letter, answer in zip("abcde", [answer_B6_Q2a, answer_B6_Q2b, answer_B6_Q2c, answer_B6_Q2d, answer_B6_Q2e]):
    wb.check(f"B6.Q2{letter}", answer)

**B6.Q3 — Dépendance partielle et courbes ICE : ce que montre la moyenne, ce qu'elle cache**

In [ ]:
# B6.Q3: the values you found on paper
answer_B6_Q3a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q3b = ...  # b) a whole number
answer_B6_Q3c = ...  # c) a number, 2 decimals
answer_B6_Q3d = ...  # d) True or False
answer_B6_Q3e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q3a, answer_B6_Q3b, answer_B6_Q3c, answer_B6_Q3d, answer_B6_Q3e]):
    wb.check(f"B6.Q3{letter}", answer)

**B6.Q4 — Les quatre propriétés de Shapley : efficacité, symétrie, joueur nul, additivité**

In [ ]:
# B6.Q4: the values you found on paper
answer_B6_Q4a = ...  # a) a number, 2 decimals
answer_B6_Q4b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q4c = ...  # c) a number, 1 decimal
answer_B6_Q4d = ...  # d) True or False
answer_B6_Q4e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B6_Q4a, answer_B6_Q4b, answer_B6_Q4c, answer_B6_Q4d, answer_B6_Q4e]):
    wb.check(f"B6.Q4{letter}", answer)

**B6.Q5 — Changer le fond (background) change les valeurs SHAP : vrai ou faux**

In [ ]:
# B6.Q5: the values you found on paper
answer_B6_Q5a = ...  # a) True or False
answer_B6_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q5c = ...  # c) a whole number
answer_B6_Q5d = ...  # d) a whole number
answer_B6_Q5e = ...  # e) True or False
answer_B6_Q5f = ...  # f) a whole number

for letter, answer in zip("abcdef", [answer_B6_Q5a, answer_B6_Q5b, answer_B6_Q5c, answer_B6_Q5d, answer_B6_Q5e, answer_B6_Q5f]):
    wb.check(f"B6.Q5{letter}", answer)

**B6.Q6 — Features corrélées : ce que SHAP répartit et ce qu'il ne dit pas**

In [ ]:
# B6.Q6: the values you found on paper
answer_B6_Q6a = ...  # a) a whole number
answer_B6_Q6b = ...  # b) a whole number
answer_B6_Q6c = ...  # c) True or False
answer_B6_Q6d = ...  # d) the letter of your choice, e.g. "E"
answer_B6_Q6e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q6a, answer_B6_Q6b, answer_B6_Q6c, answer_B6_Q6d, answer_B6_Q6e]):
    wb.check(f"B6.Q6{letter}", answer)

**B6.Q7 — Gradient, gradient × entrée, gradients intégrés : pourquoi une ligne de référence ?**

In [ ]:
# B6.Q7: the values you found on paper
answer_B6_Q7a = ...  # a) a whole number
answer_B6_Q7b = ...  # b) a whole number
answer_B6_Q7c = ...  # c) a whole number
answer_B6_Q7d = ...  # d) True or False
answer_B6_Q7e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q7a, answer_B6_Q7b, answer_B6_Q7c, answer_B6_Q7d, answer_B6_Q7e]):
    wb.check(f"B6.Q7{letter}", answer)

**B6.Q8 — Grad-CAM : pourquoi la dernière couche convolutive ?**

In [ ]:
# B6.Q8: the values you found on paper
answer_B6_Q8a = ...  # a) a whole number
answer_B6_Q8b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q8c = ...  # c) a whole number
answer_B6_Q8d = ...  # d) True or False
answer_B6_Q8e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q8a, answer_B6_Q8b, answer_B6_Q8c, answer_B6_Q8d, answer_B6_Q8e]):
    wb.check(f"B6.Q8{letter}", answer)

**B6.Q9 — Une carte de saillance qui ne dépend pas du modèle : que prouve le test de randomisation ?**

In [ ]:
# B6.Q9: the values you found on paper
answer_B6_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q9b = ...  # b) True or False
answer_B6_Q9c = ...  # c) the letter of your choice, e.g. "E"
answer_B6_Q9d = ...  # d) True or False
answer_B6_Q9e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B6_Q9a, answer_B6_Q9b, answer_B6_Q9c, answer_B6_Q9d, answer_B6_Q9e]):
    wb.check(f"B6.Q9{letter}", answer)

**B6.Q10 — Sources de biais : données, labels, variables de substitution, boucles de rétroaction**

In [ ]:
# B6.Q10: the values you found on paper
answer_B6_Q10a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q10c = ...  # c) the letter of your choice, e.g. "E"
answer_B6_Q10d = ...  # d) True or False
answer_B6_Q10e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q10a, answer_B6_Q10b, answer_B6_Q10c, answer_B6_Q10d, answer_B6_Q10e]):
    wb.check(f"B6.Q10{letter}", answer)

**B6.Q11 — Parité démographique, égalité des chances, calibration : peut-on tout avoir ?**

In [ ]:
# B6.Q11: the values you found on paper
answer_B6_Q11a = ...  # a) a number, 2 decimals
answer_B6_Q11b = ...  # b) True or False
answer_B6_Q11c = ...  # c) True or False
answer_B6_Q11d = ...  # d) a number, 2 decimals
answer_B6_Q11e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_Q11a, answer_B6_Q11b, answer_B6_Q11c, answer_B6_Q11d, answer_B6_Q11e]):
    wb.check(f"B6.Q11{letter}", answer)

**B6.Q12 — AI Act : dans quelle catégorie de risque tombe un scoring de crédit ?**

In [ ]:
# B6.Q12: the values you found on paper
answer_B6_Q12a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_Q12b = ...  # b) the letter of your choice, e.g. "E"
answer_B6_Q12c = ...  # c) the letter of your choice, e.g. "E"
answer_B6_Q12d = ...  # d) True or False
answer_B6_Q12e = ...  # e) the letter of your choice, e.g. "E"
answer_B6_Q12f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B6_Q12a, answer_B6_Q12b, answer_B6_Q12c, answer_B6_Q12d, answer_B6_Q12e, answer_B6_Q12f]):
    wb.check(f"B6.Q12{letter}", answer)

**B6.R1 — Visualiser ce qu'a appris un CNN, et l'attaque FGSM**

In [ ]:
# B6.R1: the values you found on paper
answer_B6_R1a = ...  # a) a list of three numbers, 2 decimals
answer_B6_R1b = ...  # b) True or False
answer_B6_R1c = ...  # c) the letter of your choice, e.g. "E"
answer_B6_R1d = ...  # d) a number, 2 decimals
answer_B6_R1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B6_R1a, answer_B6_R1b, answer_B6_R1c, answer_B6_R1d, answer_B6_R1e]):
    wb.check(f"B6.R1{letter}", answer)

**B6.R2 — feature_importances_ contre importance par permutation**

In [ ]:
# B6.R2: the values you found on paper
answer_B6_R2a = ...  # a) the letter of your choice, e.g. "E"
answer_B6_R2b = ...  # b) True or False
answer_B6_R2c = ...  # c) a whole number
answer_B6_R2d = ...  # d) the letter of your choice, e.g. "E"
answer_B6_R2e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B6_R2a, answer_B6_R2b, answer_B6_R2c, answer_B6_R2d, answer_B6_R2e]):
    wb.check(f"B6.R2{letter}", answer)

**B6.R3 — TPR, FPR et precision à partir d'une matrice de confusion**

In [ ]:
# B6.R3: the values you found on paper
answer_B6_R3a = ...  # a) a number, 2 decimals
answer_B6_R3b = ...  # b) a number, 2 decimals
answer_B6_R3c = ...  # c) a number, 3 decimals
answer_B6_R3d = ...  # d) a number, 3 decimals
answer_B6_R3e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B6_R3a, answer_B6_R3b, answer_B6_R3c, answer_B6_R3d, answer_B6_R3e]):
    wb.check(f"B6.R3{letter}", answer)

**Ex B6.1 — Valeurs de Shapley exactes à la main : trois features, huit coalitions**

In [ ]:
# Ex B6.1: the values you found on paper
answer_B6_1a = ...  # a) a whole number
answer_B6_1b = ...  # b) a whole number
answer_B6_1c = ...  # c) a whole number
answer_B6_1d = ...  # d) a number, 2 decimals
answer_B6_1e = ...  # e) a number, 1 decimal
answer_B6_1f = ...  # f) a whole number
answer_B6_1g = ...  # g) a number, 1 decimal
answer_B6_1h = ...  # h) True or False
answer_B6_1i = ...  # i) a whole number
answer_B6_1j = ...  # j) a number, 1 decimal
answer_B6_1k = ...  # k) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefghijk", [answer_B6_1a, answer_B6_1b, answer_B6_1c, answer_B6_1d, answer_B6_1e, answer_B6_1f, answer_B6_1g, answer_B6_1h, answer_B6_1i, answer_B6_1j, answer_B6_1k]):
    wb.check(f"B6.1{letter}", answer)

**Ex B6.3 — Gradients intégrés à la main sur f(x) = x₁² + 3x₂ et vérification de la complétude**

In [ ]:
# Ex B6.3: the values you found on paper
answer_B6_3a = ...  # a) a list of two numbers
answer_B6_3b = ...  # b) a list of two numbers
answer_B6_3c = ...  # c) a whole number
answer_B6_3d = ...  # d) a list of two numbers
answer_B6_3e = ...  # e) a number, 1 decimal
answer_B6_3f = ...  # f) a number, 1 decimal
answer_B6_3g = ...  # g) a number, 1 decimal
answer_B6_3h = ...  # h) a whole number
answer_B6_3i = ...  # i) a whole number
answer_B6_3j = ...  # j) a whole number
answer_B6_3k = ...  # k) a number, 2 decimals
answer_B6_3l = ...  # l) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefghijkl", [answer_B6_3a, answer_B6_3b, answer_B6_3c, answer_B6_3d, answer_B6_3e, answer_B6_3f, answer_B6_3g, answer_B6_3h, answer_B6_3i, answer_B6_3j, answer_B6_3k, answer_B6_3l]):
    wb.check(f"B6.3{letter}", answer)

**Ex B6.4 — Grad-CAM à la main : deux cartes 3×3 et leurs gradients**

In [ ]:
# Ex B6.4: the values you found on paper
answer_B6_4a = ...  # a) a number, 1 decimal
answer_B6_4b = ...  # b) a number, 1 decimal
answer_B6_4c = ...  # c) a number, 1 decimal
answer_B6_4d = ...  # d) a whole number
answer_B6_4e = ...  # e) a number, 2 decimals
answer_B6_4f = ...  # f) the letter of your choice, e.g. "E"
answer_B6_4g = ...  # g) a number, 2 decimals
answer_B6_4h = ...  # h) the letter of your choice, e.g. "E"
answer_B6_4i = ...  # i) True or False
answer_B6_4j = ...  # j) a whole number

for letter, answer in zip("abcdefghij", [answer_B6_4a, answer_B6_4b, answer_B6_4c, answer_B6_4d, answer_B6_4e, answer_B6_4f, answer_B6_4g, answer_B6_4h, answer_B6_4i, answer_B6_4j]):
    wb.check(f"B6.4{letter}", answer)

**Ex B6.5 — Audit à la main : deux groupes, deux matrices de confusion, cinq indicateurs**

In [ ]:
# Ex B6.5: the values you found on paper
answer_B6_5a = ...  # a) a number, 2 decimals
answer_B6_5b = ...  # b) a number, 3 decimals
answer_B6_5c = ...  # c) a number, 2 decimals
answer_B6_5d = ...  # d) a number, 3 decimals
answer_B6_5e = ...  # e) a number, 3 decimals
answer_B6_5f = ...  # f) a number, 2 decimals
answer_B6_5g = ...  # g) a number, 2 decimals
answer_B6_5h = ...  # h) a number, 2 decimals
answer_B6_5i = ...  # i) True or False
answer_B6_5j = ...  # j) True or False
answer_B6_5k = ...  # k) a whole number

for letter, answer in zip("abcdefghijk", [answer_B6_5a, answer_B6_5b, answer_B6_5c, answer_B6_5d, answer_B6_5e, answer_B6_5f, answer_B6_5g, answer_B6_5h, answer_B6_5i, answer_B6_5j, answer_B6_5k]):
    wb.check(f"B6.5{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer à la main les valeurs de Shapley d'un petit jeu, avec un fond, et dire ce que le fond change ?
2. Sais-tu calculer des gradients intégrés et une carte Grad-CAM à la main, et vérifier la complétude ?
3. Sais-tu auditer un classifieur par groupe (taux de sélection, TPR, FPR, precision, écarts) et expliquer pourquoi tous les critères ne peuvent pas tenir ensemble ?
4. Sais-tu dire ce qu'un scoring de crédit implique au regard du RGPD et de l'AI Act ?

**Pour aller plus loin** : l'article de Lundberg et Lee et le livre en ligne de C. Molnar, cités dans la fiche. La suite : le bonus B7, le MLOps.

*Les exercices de code B6.12 à B6.29 (modèles tabulaires, réseaux, équité) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B6` pour obtenir le notebook complet.*